# 1. Load and validate the cleaned dataset

Load the prepared hERG data and check that canonical SMILES are present and unique and that the blocking labels are binary. This confirms the dataset is ready for scaffold analysis and splitting.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
requirements_path = project_root / "requirements.txt"

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%pip install -r "{requirements_path}" 

import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from src.evaluation import get_scaffold


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Locate the project directory.
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

df_clf = pd.read_csv(
    project_root / "data" / "processed" / "herg_clean.csv"
)

# Check that cleaning has been completed.
assert df_clf["Canonical_smiles"].notna().all()
assert df_clf["Canonical_smiles"].is_unique
assert df_clf["Blocking_status"].isin([0, 1]).all()

df_clf["Blocking_status"] = df_clf["Blocking_status"].astype(int)

print("Number of molecules:", len(df_clf))

Number of molecules: 645


# 2. Examine the scaffold distribution

Assign each molecule a Bemis–Murcko scaffold and inspect the most common scaffold groups. This shows how many compounds share a structural core and motivates keeping related scaffolds together when splitting the data.

In [3]:

df_clf["scaffold"] = df_clf["Canonical_smiles"].apply(get_scaffold)

assert df_clf["scaffold"].notna().all(), "Invalid structures to inspect."

print("Number of groups:", df_clf["scaffold"].nunique())

# Inspect the largest groups.
display(df_clf["scaffold"].value_counts().head(10))


Number of groups: 439


[11:43:59] WARNING: not removing hydrogen atom without neighbors
[11:43:59] WARNING: not removing hydrogen atom without neighbors


scaffold
c1ccccc1                                              32
O=C(NC1CCN(Cc2ccc3c(c2)OCO3)CC1)c1cc(=O)c2ccccc2o1    15
c1ccc(Cc2ccccc2)cc1                                   12
c1ccc(-c2[nH]c3ccccc3c2[C@H]2CCC[NH2+]C2)cc1          10
c1ccc(-n2ccc3ccccc32)cc1                               8
O=C(NC1CCN(Cc2ccccc2)CC1)c1cc(=O)c2ccccc2o1            8
O=C1NCCN1CCN1CCC(c2cn(-c3ccccc3)c3ccccc23)CC1          7
O=C1NCCN1CC[NH+]1CCC(c2cn(-c3ccccc3)c3ccccc23)CC1      7
c1ccc(Cc2ccccn2)cc1                                    6
c1ccc2c(c1)Nc1ccccc1S2                                 6
Name: count, dtype: int64

### Interpreting the most common scaffolds

The table shows that the molecules are not all structurally unrelated: some share the same underlying molecular core.
The Bemis–Murcko method  keeps a molecule's ring systems and the chains that connect those rings, while removing most groups attached around the outside. Compounds with the same remaining framework are therefore placed in the same scaffold group, even if their attached side groups differ.

This pattern matters for modeling: if compounds with the same core were randomly placed in both training and test sets, the model could be evaluated on structures very similar to ones it had already seen. The scaffold-based split below keeps each group together, so the test set better measures how well the model handles compounds with scaffold frameworks absent from training. The counts also reveal that some scaffolds form larger groups than others, which can make a balanced split more challenging.

# 3. Create scaffold-aware data splits

Reserve a scaffold-disjoint test set for final evaluation, then divide the training set into five scaffold-aware cross-validation folds for model selection. Grouping by scaffold reduces the chance that closely related molecular cores appear in both fitting and validation data.

In [4]:
# Configure a five-fold cross-validation split.
# Keep compounds with the same scaffold in the same fold to avoid related structures
# appearing in both the training and test sets.
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X=df_clf,
        y=df_clf["Blocking_status"],
        groups=df_clf["scaffold"]
    )
)
assert df_clf["scaffold"].notna().all(), "Invalid structures to inspect."

# Use the first split: one fold for testing and four for training
# The fixed random state makes the shuffled split reproducible.
# Label each molecule with its assigned partition.
df_clf["split"] = "train"
df_clf.loc[test_idx, "split"] = "test"

# Create separate datasets and reset their indexes for convenient downstream use.
train_df = df_clf.iloc[train_idx].copy().reset_index(drop=True)
test_df = df_clf.iloc[test_idx].copy().reset_index(drop=True)

In [5]:
# No group or identical structure should occur in both sets.
assert set(train_df["scaffold"]).isdisjoint(test_df["scaffold"])
assert set(train_df["Canonical_smiles"]).isdisjoint(test_df["Canonical_smiles"])

# Both sets must contain blockers and non-blockers.
assert train_df["Blocking_status"].nunique() == 2
assert test_df["Blocking_status"].nunique() == 2

# Summarize class counts and split sizes.
summary = pd.crosstab(
    df_clf["split"],
    df_clf["Blocking_status"]
).reindex(columns=[0, 1], fill_value=0)

summary.columns = ["Non-blockers", "Blockers"]
summary["Total"] = summary["Non-blockers"] + summary["Blockers"]
summary["Dataset (%)"] = 100 * summary["Total"] / len(df_clf)
summary["Blockers (%)"] = 100 * summary["Blockers"] / summary["Total"]

display(summary.round(1))
display(test_df.head())
display(train_df.head())

,Non-blockers,Blockers,Total,Dataset (%),Blockers (%)
split,,,,,
test,50,76,126,19.5,60.3
train,151,368,519,80.5,70.9


,Drug_ID,SMILES,Blocking_status,Canonical_smiles,scaffold,split
0,LY-97241,CCCCCCCN(CC)CCCCc1ccc([N+](=O)[O-])cc1,1,CCCCCCCN(CC)CCCCc1ccc([N+](=O)[O-])cc1,c1ccccc1,test
1,VANOXERINE HYDROCHLORIDE,Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,1,Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,c1ccc(CCCN2CCN(CCOC(c3ccccc3)c3ccccc3)CC2)cc1,test
2,DOFETILIDE,C[NH+](CCOc1ccc(NS(C)(=O)=O)cc1)CCc1ccc(NS(C)(...,1,C[NH+](CCOc1ccc(NS(C)(=O)=O)cc1)CCc1ccc(NS(C)(...,c1ccc(CC[NH2+]CCOc2ccccc2)cc1,test
3,BMCL20031829_15,CN(C)Cc1cc2cc(Cl)ccc2n1-c1ccc(F)cc1,1,CN(C)Cc1cc2cc(Cl)ccc2n1-c1ccc(F)cc1,c1ccc(-n2ccc3ccccc32)cc1,test
4,CLOFILIUM,CCCCCCC[N+](CC)(CC)CCCCc1ccc(Cl)cc1,1,CCCCCCC[N+](CC)(CC)CCCCc1ccc(Cl)cc1,c1ccccc1,test


,Drug_ID,SMILES,Blocking_status,Canonical_smiles,scaffold,split
0,DEMETHYLASTEMIZOLE,Oc1ccc(CCN2CCC(Nc3nc4ccccc4n3Cc3ccc(F)cc3)CC2)cc1,1,Oc1ccc(CCN2CCC(Nc3nc4ccccc4n3Cc3ccc(F)cc3)CC2)cc1,c1ccc(CCN2CCC(Nc3nc4ccccc4n3Cc3ccccc3)CC2)cc1,train
1,GBR-12909,Fc1ccc(C(OCC[NH+]2CC[NH+](CCCc3ccccc3)CC2)c2cc...,1,Fc1ccc(C(OCC[NH+]2CC[NH+](CCCc3ccccc3)CC2)c2cc...,c1ccc(CCC[NH+]2CC[NH+](CCOC(c3ccccc3)c3ccccc3)...,train
2,CLOFILIUM PHOSPHATE,CCCCCCC[N+](CC)(CC)CCCCc1ccc(Cl)cc1.CCCCCCC[N+...,1,CCCCCCC[N+](CC)(CC)CCCCc1ccc(Cl)cc1.CCCCCCC[N+...,c1ccccc1.c1ccccc1.c1ccccc1,train
3,FLUSPIRILENE,O=C1NCN(c2ccccc2)C12CC[NH+](CCCC(c1ccc(F)cc1)c...,1,O=C1NCN(c2ccccc2)C12CC[NH+](CCCC(c1ccc(F)cc1)c...,O=C1NCN(c2ccccc2)C12CC[NH+](CCCC(c1ccccc1)c1cc...,train
4,BENPERIDOL,O=C(CCCN1CCC(n2c(=O)[nH]c3ccccc32)CC1)c1ccc(F)cc1,1,O=C(CCCN1CCC(n2c(=O)[nH]c3ccccc32)CC1)c1ccc(F)cc1,O=C(CCCN1CCC(n2c(=O)[nH]c3ccccc32)CC1)c1ccccc1,train


In [6]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_splits = list(
    cv.split(
        X=train_df,
        y=train_df["Blocking_status"],
        groups=train_df["scaffold"]
    )
)

train_df["cv_fold"] = -1
fold_summary = []

for fold, (fit_idx, valid_idx) in enumerate(cv_splits):
    fit_data = train_df.iloc[fit_idx]
    valid_data = train_df.iloc[valid_idx]

    # Check group separation.
    assert set(fit_data["scaffold"]).isdisjoint(valid_data["scaffold"])

    # Both classes must be present in training and validation.
    assert fit_data["Blocking_status"].nunique() == 2, (
        f"Only one class in training fold {fold}."
    )
    assert valid_data["Blocking_status"].nunique() == 2, (
        f"Only one class in validation fold {fold}."
    )

    # Record the fold where each molecule is used for validation.
    train_df.loc[valid_idx, "cv_fold"] = fold

    fold_summary.append({
        "Fold": fold,
        "Training molecules": len(fit_data),
        "Training blockers": int(fit_data["Blocking_status"].sum()),
        "Validation molecules": len(valid_data),
        "Validation blockers": int(valid_data["Blocking_status"].sum())
    })

assert train_df["cv_fold"].ge(0).all()

display(pd.DataFrame(fold_summary))

,Fold,Training molecules,Training blockers,Validation molecules,Validation blockers
0,0,409,294,110,74
1,1,412,295,107,73
2,2,418,293,101,75
3,3,416,294,103,74
4,4,421,296,98,72


In [7]:
# Map validation folds back to the full dataset.
fold_mapping = train_df.set_index("Canonical_smiles")["cv_fold"]

df_clf["cv_fold"] = (
    df_clf["Canonical_smiles"]
    .map(fold_mapping)
    .astype("Int64")
)

display(df_clf.head())

# Test molecules intentionally have no CV fold.
split_dir = project_root / "data" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

df_clf.to_csv(
    split_dir / "herg_scaffold_split.csv",
    index=False
)


,Drug_ID,SMILES,Blocking_status,Canonical_smiles,scaffold,split,cv_fold
0,DEMETHYLASTEMIZOLE,Oc1ccc(CCN2CCC(Nc3nc4ccccc4n3Cc3ccc(F)cc3)CC2)cc1,1,Oc1ccc(CCN2CCC(Nc3nc4ccccc4n3Cc3ccc(F)cc3)CC2)cc1,c1ccc(CCN2CCC(Nc3nc4ccccc4n3Cc3ccccc3)CC2)cc1,train,4
1,GBR-12909,Fc1ccc(C(OCC[NH+]2CC[NH+](CCCc3ccccc3)CC2)c2cc...,1,Fc1ccc(C(OCC[NH+]2CC[NH+](CCCc3ccccc3)CC2)c2cc...,c1ccc(CCC[NH+]2CC[NH+](CCOC(c3ccccc3)c3ccccc3)...,train,2
2,LY-97241,CCCCCCCN(CC)CCCCc1ccc([N+](=O)[O-])cc1,1,CCCCCCCN(CC)CCCCc1ccc([N+](=O)[O-])cc1,c1ccccc1,test,<NA>
3,CLOFILIUM PHOSPHATE,CCCCCCC[N+](CC)(CC)CCCCc1ccc(Cl)cc1.CCCCCCC[N+...,1,CCCCCCC[N+](CC)(CC)CCCCc1ccc(Cl)cc1.CCCCCCC[N+...,c1ccccc1.c1ccccc1.c1ccccc1,train,3
4,FLUSPIRILENE,O=C1NCN(c2ccccc2)C12CC[NH+](CCCC(c1ccc(F)cc1)c...,1,O=C1NCN(c2ccccc2)C12CC[NH+](CCCC(c1ccc(F)cc1)c...,O=C1NCN(c2ccccc2)C12CC[NH+](CCCC(c1ccccc1)c1cc...,train,3


### Split results

The 645 molecules form 439 scaffold groups. The reproducible holdout split contains 519 training molecules (80.5%) and 126 test molecules (19.5%), with no scaffold shared between the sets. The split aims to preserve the blocker/non-blocker proportions, but the scaffold constraint means they are not identical: blockers make up 70.9% of training and 60.3% of test.

For model selection, the training set was further divided into five scaffold-aware cross-validation folds. Each fold serves once as validation while the other four are used for fitting, with scaffolds kept separate. The held-out test set remains untouched for final evaluation.



# 4. Generate and inspect molecular descriptors

Calculate 210 RDKit descriptors for each molecule in the training and test sets, keeping feature rows aligned with their labels. Check the matrix dimensions and identify missing descriptor values so they can be handled appropriately during model training.

In [8]:
from src.features import compute_descriptors

# Extract target labels.
y_train = train_df["Blocking_status"].to_numpy()
y_test = test_df["Blocking_status"].to_numpy()

# Calculate molecular features.
X_train_desc = compute_descriptors(train_df["Canonical_smiles"])
X_test_desc = compute_descriptors(test_df["Canonical_smiles"])


print("Training descriptors:", X_train_desc.shape)
print("Test descriptors:", X_test_desc.shape)

display(X_train_desc.head())
display(X_test_desc.head())

[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not removing hydrogen atom without neighbors
[11:44:05] WARNING: not r

Training descriptors: (519, 210)
Test descriptors: (126, 210)


,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,13.360182,13.360182,0.217816,-0.217816,0.418965,15.181818,444.554,415.322,444.232540,170,...,0,0,0,0,0,0,0,0,0,0
1,13.400224,13.400224,0.272972,-0.323611,0.483846,18.515152,452.589,418.317,452.262823,174,...,0,0,0,0,0,0,0,0,0,0
2,8.548611,8.548611,0.837475,-5.388889,0.032352,11.824324,1111.931,1000.043,1109.737779,428,...,0,0,0,0,0,0,0,0,12,0
3,13.504638,13.504638,0.098069,-0.448798,0.543063,22.085714,476.591,444.335,476.250796,182,...,0,0,0,0,0,0,0,0,0,0
4,12.948477,12.948477,0.037459,-0.325252,0.661510,15.892857,381.451,357.259,381.185255,146,...,0,0,0,0,0,0,0,0,0,0


,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,10.625119,10.625119,0.174983,-0.343833,0.284606,11.086957,320.477,288.221,320.246378,130,...,0,0,0,0,0,0,0,0,4,0
1,13.394919,13.394919,0.279402,-0.342152,0.415758,15.242424,450.573,418.317,450.248270,174,...,0,0,0,0,0,0,0,0,0,0
2,11.221609,11.221609,0.500322,-3.282513,0.471730,12.931034,442.583,414.359,442.146489,160,...,0,2,0,0,0,0,0,0,0,0
3,13.170886,13.170886,0.229245,-0.229245,0.692104,11.476190,302.780,286.652,302.098604,108,...,0,0,0,0,0,0,0,0,0,0
4,5.948063,5.948063,0.837475,0.837475,0.284619,11.826087,338.987,301.691,338.260904,132,...,0,0,0,0,0,0,0,0,4,0


### Descriptor checks

Confirm that the feature matrices align with the target labels and share the same descriptor columns. Inspect missing values in the training descriptors; any imputation strategy should be fitted on training data only.

In [9]:
# Check that features and labels have matching lengths.
assert len(X_train_desc) == len(y_train)
assert len(X_test_desc) == len(y_test)

# Check that both datasets contain the same descriptor columns.
assert X_train_desc.columns.equals(X_test_desc.columns)

# Count missing values in each training descriptor.
missing_values = X_train_desc.isna().sum()

print(
    "Descriptors with missing training values:",
    (missing_values > 0).sum()
)

display(missing_values[missing_values > 0])

Descriptors with missing training values: 8


BCUT2D_MWHI       4
BCUT2D_MWLOW      4
BCUT2D_CHGHI      4
BCUT2D_CHGLO      4
BCUT2D_LOGPHI     4
BCUT2D_LOGPLOW    4
BCUT2D_MRHI       4
BCUT2D_MRLOW      4
dtype: int64

In [10]:
# Identify molecules with at least one missing descriptor.
missing_mask = X_train_desc.isna().any(axis=1)

print("Molecules affected:", missing_mask.sum())

display(
    train_df.loc[
        missing_mask,
        ["Canonical_smiles", "Blocking_status"]
    ]
)

Molecules affected: 4


,Canonical_smiles,Blocking_status
445,O=C(O)CN(CCN(CC(=O)O)CC(=O)O)CC(=O)O.[Cu+2],0
446,O=C(O)CN(CCN(CC(=O)O)CC(=O)O)CC(=O)O.[Mn+2],0
452,O=C(O)CCC(O)=C1S(=O)(=O)OCCOS1(=O)=O.[H-].[Na+],0
453,CC[C@@](O)(C(=O)O)c1cc2n(c(=O)c1CO)Cc1cc3ccccc...,0


# 5. Generate Morgan fingerprints

Create radius-2 Morgan fingerprints with 2,048 bits for the training and test molecules. Verify that fingerprints preserve molecule order and contain only binary values.

In [11]:
from src.features import compute_morgan

# Calculate fingerprints in the same molecule order as the descriptors.
X_train_morgan = compute_morgan(train_df["Canonical_smiles"])
X_test_morgan = compute_morgan(test_df["Canonical_smiles"])

print("Training fingerprints:", X_train_morgan.shape)
print("Test fingerprints:", X_test_morgan.shape)

display(X_train_morgan.head())

[11:44:07] WARNING: not removing hydrogen atom without neighbors
[11:44:07] WARNING: not removing hydrogen atom without neighbors


Training fingerprints: (519, 2048)
Test fingerprints: (126, 2048)


,0,1,2,3,4,5,6,7,8,9,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,0


### Fingerprint checks

In [12]:
# Check alignment with the descriptor matrices.
assert X_train_morgan.index.equals(X_train_desc.index)
assert X_test_morgan.index.equals(X_test_desc.index)

# Check that all fingerprint values are binary.
assert X_train_morgan.isin([0, 1]).all().all()
assert X_test_morgan.isin([0, 1]).all().all()

### Summary before model fitting

Before fitting the classification models, the cleaned hERG dataset was checked to ensure that canonical SMILES are present and unique and that `Blocking_status` contains only the binary labels 0 and 1. Molecules were divided into training and held-out test sets using a scaffold-aware split: the 519 training molecules and 126 test molecules do not share Bemis–Murcko scaffolds. The training set was further organized into five scaffold-aware cross-validation folds for model selection, leaving the test set aside for final evaluation.

Two molecular representations were then generated for both sets, preserving the molecule order and alignment with the target labels:

- **RDKit descriptors:** 210 numerical descriptors per molecule (519 × 210 for training; 126 × 210 for test).
- **Morgan fingerprints:** 2,048-bit fingerprints with radius 2 (519 × 2,048 for training; 126 × 2,048 for test).

The feature matrices were checked for matching dimensions and row alignment; the Morgan fingerprints were also verified to contain only binary values. The descriptor matrices contain missing values in 8 descriptor columns affecting 4 training molecules, so these values should be handled using a strategy fitted on the training data before model fitting.